#  귀농 유형별 인프라 기반 주거지 추출

인프라 데이터를 기반으로 AHP 방식의 가중치를 적용해  
귀농 유형별로 적합한 주거지를 선별하고, 상위 10개 후보지를 추출하는 과정을 담고 있습니다.


In [1]:
# 필수 라이브러리 설치
!pip install koreanize_matplotlib factor_analyzer yellowbrick
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import platform
from collections import defaultdict
import koreanize_matplotlib

# 한글 폰트 설정
from matplotlib import rc
if platform.system() == 'Windows':
    rc('font', family='Malgun Gothic')
elif platform.system() == 'Darwin':
    rc('font', family='AppleGothic')
else:
    rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 757.7 kB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 59.2 MB/s eta 0:00:00
  Created wheel for factor_analyzer: filename=factor_analyzer-0.5.1-py2.py3-none-any.whl size=42655 sha256=9486630b4a7bf8056bc4bcd073a99954a606b37c6f0e77086ab529bc59b2ab9d
  Stored in directory: /root/.cache/pip/wheels/fa/f7/53/a55a8a56668a6fe0199e0e02b6e0ae3007ec35cdf6e4c25df7
Successfully built factor_analyzer


## 1.데이터 불러오기

- infra_category_clustered.csv: 클러스터링된 인프라 점수 데이터  
- AHP_weights.csv: 귀농 유형별 가중치 데이터  


In [2]:
# 코랩 전용
from google.colab import drive
import os
drive.mount('/content/drive')
os.chdir("/content/drive/MyDrive/비어플/dataset")
print("현재 작업 디렉토리:", os.getcwd())

Mounted at /content/drive
현재 작업 디렉토리: /content/drive/MyDrive/비어플/dataset


In [3]:
infra_df = pd.read_csv("infra_category_clustered.csv")
weights_df = pd.read_csv("AHP_weights.csv").set_index("유형")

## 2.클러스터-유형 매핑 설정

각 귀농 유형이 선호하는 클러스터 번호를 지정합니다.


In [4]:
type_to_cluster = pd.read_csv("cluster_matching_result.csv").set_index("유형")["클러스터"].to_dict()

cluster_to_types = defaultdict(list)
for user_type, cluster_num in type_to_cluster.items():
    cluster_to_types[cluster_num].append(user_type)

## 3.가중합 점수 계산

각 클러스터의 데이터에 대해, 해당 유형의 AHP 가중치를 적용하여  
가중합 점수를 계산합니다.

In [5]:
scored_rows = []

for cluster_num, user_type_list in cluster_to_types.items():
    cluster_data = infra_df[(infra_df["cluster"] == cluster_num) & (infra_df["농업_매물"] >= 0)]

    for user_type in user_type_list:
        weights = weights_df.loc[user_type]
        temp = cluster_data.copy()
        temp["적용유형"] = user_type
        temp["가중합점수"] = temp[weights.index].dot(weights)
        scored_rows.append(temp)

scored_df = pd.concat(scored_rows, ignore_index=True)
scored_df.head()

,address,농업_지원,농업_매물,교통,교육,편의,서비스,cluster,적용유형,가중합점수
0,경상북도 포항남구 동해면 도구리,-0.080621,0.101177,0.920533,1.749410,1.781770,1.672603,0,은퇴생계형,1.017581
1,경상남도 통영시 용남면 화삼리,-0.237935,0.085655,0.407946,0.737601,1.933758,1.650385,0,은퇴생계형,0.865373
2,경상남도 통영시 문화동,-0.327605,1.285648,0.363965,2.620217,2.337710,2.009291,0,은퇴생계형,1.201660
3,경상남도 통영시 태평동,-0.322352,0.968651,0.407843,2.757525,2.471749,2.023847,0,은퇴생계형,1.229631
4,경상남도 통영시 중앙동,-0.345359,0.925178,0.408973,2.597596,2.488245,2.017408,0,은퇴생계형,1.219931


## 4.분위 점수 계산

전체 데이터 분포를 기준으로 인프라 항목별 분위 점수(1~10점)를 계산합니다.

In [6]:
infra_cols = ["농업_지원", "농업_매물", "교통", "교육", "편의", "서비스"]
final_df = scored_df[["address", "cluster", "적용유형", "가중합점수"]].copy()

for col in infra_cols:
    bins = pd.qcut(infra_df[col], 10, retbins=True, duplicates="drop")[1]
    final_df[col] = pd.cut(scored_df[col], bins=bins, labels=False, include_lowest=True) + 1

final_df.head()

,address,cluster,적용유형,가중합점수,농업_지원,농업_매물,교통,교육,편의,서비스
0,경상북도 포항남구 동해면 도구리,0,은퇴생계형,1.017581,5,7,9,10,10,10
1,경상남도 통영시 용남면 화삼리,0,은퇴생계형,0.865373,5,7,8,9,10,10
2,경상남도 통영시 문화동,0,은퇴생계형,1.201660,4,9,8,10,10,10
3,경상남도 통영시 태평동,0,은퇴생계형,1.229631,4,9,8,10,10,10
4,경상남도 통영시 중앙동,0,은퇴생계형,1.219931,4,9,8,10,10,10


## 5.유형별 상위 10개 추천지 추출 및 저장

귀농 유형별로 가중합 점수가 높은 10개 추천지를 추출하고 저장합니다.


In [7]:
top10_df = final_df.groupby("적용유형").apply(
    lambda x: x.nlargest(10, "가중합점수")
).reset_index(drop=True)

<ipython-input-7-d7930a870555>:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  top10_df = final_df.groupby("적용유형").apply(


In [8]:
top10_df

,address,cluster,적용유형,가중합점수,농업_지원,농업_매물,교통,교육,편의,서비스
0,경상북도 영주시 하망동,0,가족정착형,2.193685,6,10,6,10,10,10
1,경상북도 상주시 냉림동,0,가족정착형,2.193339,8,10,10,10,10,10
2,경상북도 영주시 영주동,0,가족정착형,2.153575,6,9,7,10,10,10
3,경상북도 영주시 휴천동,0,가족정착형,2.143976,6,9,7,10,10,10
4,전라북도 전주덕진구 호성동1가,0,가족정착형,2.119683,10,9,10,10,10,10
5,경상북도 경주시 노서동,0,가족정착형,2.099816,10,9,6,10,10,10
6,경상북도 상주시 서문동,0,가족정착형,2.079938,7,9,10,10,10,10
7,경상북도 상주시 서성동,0,가족정착형,2.065518,8,9,10,10,10,10
8,경상북도 문경시 흥덕동,0,가족정착형,2.042014,9,9,10,10,10,10
9,경상북도 문경시 점촌동,0,가족정착형,2.041460,9,9,10,10,10,10


In [ ]:
top10_df.to_csv("귀농유형별_가중합_상위10.csv", index=False, encoding="utf-8-sig")